## Assignment: Object detection
- Alumno 1:
- Alumno 2:
- Alumno 3:

The goals of the assignment are:
* Put into practice acquired knowledge to detect and recognize objects of interest within a satellite image.

To address this problem, you must choose one of the following options:
*	Implement a sliding window strategy to process the whole image, and then train a classifier that determines whether each window includes or not an object of interest. In this way, you can use previous image classification model to infer the object category.
*	Build a single-stage object detection model (e.g., YOLO, SSD, RetinaNet, etc.).
*	Build a two-stage object detection model (e.g., Faster R-CNN, R-FCN, etc.).

Follow the link below to download the detection data set “xview_detection”: [https://drive.upm.es/s/P7nEf3Bygns7tbM](https://drive.upm.es/s/P7nEf3Bygns7tbM)

In [ ]:
# Install compatible versions
!pip install torch==2.1.0 torchvision==0.16.0 --upgrade
!pip install ultralytics --upgrade

In [ ]:
# All imports consolidated here
import torch
from ultralytics import YOLO

import uuid
import numpy as np
import json
import os
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.cuda.is_available()}")

In [ ]:
class GenericObject:
    """
    Generic object data.
    """
    def __init__(self):
        self.id = uuid.uuid4()
        self.bb = (-1, -1, -1, -1)
        self.category= -1
        self.score = -1

class GenericImage:
    """
    Generic image data.
    """
    def __init__(self, filename):
        self.filename = filename
        self.tile = np.array([-1, -1, -1, -1])  # (pt_x, pt_y, pt_x+width, pt_y+height)
        self.objects = list([])

    def add_object(self, obj: GenericObject):
        self.objects.append(obj)

In [ ]:
categories = {0: 'Small car', 1: 'Bus', 2: 'Truck', 3: 'Building'}

In [ ]:
# === CONFIGURATION - HPC paths ===
BASE_PATH = '/home/x467/PROJECT/xview_detection'  # Read-only dataset location
WORK_DIR = '/home/x467/x467028'  # Writable working directory
DEVICE = '0'  # GPU device

#### Training
Design and train a detector to deal with the “xview_detection” perception task.

##### Hyperparameters 

Optimization techniques: ADAM and Batch Size

In [ ]:
BATCH_SIZE = 4
LEARNING_RATE = 1e-3
CONF = 0.2  # confidence threshold
IOU = 0.5  # IoU threshold for mAP calculation
EPOCHS = 1
SIZE = 640

In [ ]:
# Load database
json_file = f'{BASE_PATH}/xview_det_train.json'
with open(json_file) as ifs:
    json_data = json.load(ifs)
ifs.close()

In [ ]:
# 1. OPTIMIZATION: Group annotations by image_id first
# This prevents checking every annotation for every image (the previous bottleneck)
ann_dict = defaultdict(list)
for ann in json_data['annotations'].values():
    ann_dict[ann['image_id']].append(ann)

counts = dict.fromkeys(categories.values(), 0)
anns = []

# 2. Iterate through images
for json_img in json_data['images'].values():
    image = GenericImage(json_img['filename'])
    image.tile = np.array([0, 0, json_img['width'], json_img['height']])
    
    # 3. Fast lookup of annotations for the current image
    current_anns = ann_dict.get(json_img['image_id'], [])
    
    for json_ann in current_anns:
        obj = GenericObject()
        # Use image_id (as in original) or ann_id depending on preference, 
        # but maintaining original logic here:
        obj.id = json_ann['image_id'] 
        obj.bb = (int(json_ann['bbox'][0]), int(json_ann['bbox'][1]), int(json_ann['bbox'][2]), int(json_ann['bbox'][3]))
        obj.category = json_ann['category_id']
        
        # Update counts and add to image
        counts[obj.category] += 1
        image.add_object(obj)
        
    anns.append(image)

print(counts)

In [ ]:
anns_train, anns_valid = train_test_split(anns, test_size=0.1, random_state=1, shuffle=True)
print(f'Training images: {len(anns_train)}')
print(f'Validation images: {len(anns_valid)}')

In [ ]:
# Create YOLO format labels in the WRITABLE working directory
def create_yolo_dataset(anns, split_name):
    """Convert annotations to YOLO format
    YOLO format: class_id x_center y_center width height (all normalized 0-1)
    """
    # Create labels in the writable working directory
    labels_dir = Path(f'{WORK_DIR}/labels/{split_name}')
    labels_dir.mkdir(parents=True, exist_ok=True)
    
    for img_ann in anns:
        # Create label file for this image
        label_path = labels_dir / (Path(img_ann.filename).stem + '.txt')
        
        with open(label_path, 'w') as f:
            img_width = img_ann.tile[2] - img_ann.tile[0]
            img_height = img_ann.tile[3] - img_ann.tile[1]
            
            for obj in img_ann.objects:
                # Get class id (0-based index)
                class_id = list(categories.keys())[list(categories.values()).index(obj.category)]
                
                # Convert from xyxy to xywh (normalized)
                x1, y1, x2, y2 = obj.bb
                x_center = ((x1 + x2) / 2) / img_width
                y_center = ((y1 + y2) / 2) / img_height
                width = (x2 - x1) / img_width
                height = (y2 - y1) / img_height
                
                # Write to file
                f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")
    
    return len(anns)

# Create datasets
print('Creating YOLO format datasets...')
num_train = create_yolo_dataset(anns_train, 'train')
num_val = create_yolo_dataset(anns_valid, 'val')
print(f'Created {num_train} training labels and {num_val} validation labels')

# Create data.yaml configuration file
# Images are in BASE_PATH (read-only), labels are in WORK_DIR (writable)
data_yaml = f"""train:
  images: {BASE_PATH}/xview_train
  labels: {WORK_DIR}/labels/train
val:
  images: {BASE_PATH}/xview_train
  labels: {WORK_DIR}/labels/val

names:
  0: Small car
  1: Bus
  2: Truck
  3: Building
"""

yaml_path = f'{WORK_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    f.write(data_yaml)

print('Created data.yaml configuration file')
print(f'Labels created in: {WORK_DIR}/labels/train/ and {WORK_DIR}/labels/val/')
print(f'Images are in: {BASE_PATH}/xview_train/')
print(f'Config file: {yaml_path}')

In [ ]:
# Initialize YOLOv11 model
model = YOLO('yolo11m.pt')
print("Model loaded!")

In [ ]:
print('Training YOLOv11 model (1 epoch for testing)')

# Train the model - Ultralytics handles everything automatically!
results = model.train(
    data=f'{WORK_DIR}/data.yaml',      # path to dataset config
    epochs=EPOCHS,                           # just 1 epoch for testing
    imgsz=SIZE,                          # image size
    batch=BATCH_SIZE,                   # batch size
    lr0=LEARNING_RATE,                  # initial learning rate
    device=DEVICE,                      # device from config
    project=WORK_DIR,                   # save directory
    name='yolov11_test',                # experiment name
    
    # Data augmentation
    mosaic=1.0,                         # mosaic augmentation probability
    fliplr=0.5,                         # horizontal flip probability
    
    # Training optimizations
    patience=15,                        # early stopping patience
    save=True,                          # save checkpoints
    cache=False,                        # cache images for faster training
    
    # Validation
    val=True,                           # validate during training
    plots=True,                         # save training plots
)

print('Training complete!')
print(f'Best model saved at: {results.save_dir}')

#### Validation
Compute validation metrics.

In [ ]:
# Load the best trained model
model = YOLO(f'{WORK_DIR}/yolov11_test/weights/best.pt')

# Run validation - Ultralytics computes all metrics automatically
print('Running validation...')
metrics = model.val(
    data=f'{WORK_DIR}/data.yaml',
    imgsz=SIZE,
    batch=1,
    conf=CONF,
    iou=IOU,
    device=DEVICE,
    plots=True,    # save plots
)

# Print key metrics
print('\n=== Validation Metrics ===')
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

# Per-class metrics
print('\n=== Per-Class Metrics ===')
for i, category in categories.items():
    print(f"{category}:")
    print(f"  AP50: {metrics.box.ap50[i]:.3f}")
    print(f"  Precision: {metrics.box.p[i]:.3f}")
    print(f"  Recall: {metrics.box.r[i]:.3f}")

#### Testing
Try to improve the results provided in the competition.

In [ ]:
anns = []
test_path = os.path.join(BASE_PATH, 'xview_test')

for (dirpath, dirnames, filenames) in os.walk(test_path):
    for filename in filenames:
        if filename.endswith(('.jpg', '.png', '.jpeg', '.tif', '.tiff')):  # only image files
            rel_path = os.path.relpath(dirpath, BASE_PATH)
            image = GenericImage(os.path.join(rel_path, filename))
            image.tile = np.array([0, 0, SIZE, SIZE])
            anns.append(image)
            
print(f'Number of testing images: {len(anns)}')

In [ ]:
# Load the best trained model
model = YOLO(f'{WORK_DIR}/yolov11_test/weights/best.pt')

# Process test images
predictions_data = {"images": {}, "annotations": {}, "categories": {}}
predictions_data["categories"] = categories

imgs_idx, annos_idx = 0, 0

for ann in tqdm(anns):
    # Run inference on the image
    img_path = os.path.join(BASE_PATH, ann.filename)
    results = model.predict(
        source=img_path,
        imgsz=SIZE,
        conf=CONF,
        iou=IOU,
        device=DEVICE,
        verbose=False,
    )[0]  # get first result
    
    # Extract predictions
    boxes = results.boxes
    num_objects = len(boxes)
    
    # Add image info
    image_data = {
        "image_id": ann.filename.split('/')[-1],
        "filename": ann.filename,
        "num_objects": num_objects,
        "width": SIZE,
        "height": SIZE
    }
    predictions_data["images"][imgs_idx] = image_data
    imgs_idx += 1
    
    # Add annotations (detections)
    for i in range(num_objects):
        box = boxes[i]
        xyxy = box.xyxy[0].cpu().numpy()  # get box coordinates
        conf = float(box.conf[0].cpu().numpy())  # get confidence
        cls = int(box.cls[0].cpu().numpy())  # get class id
        
        # Get category name
        category_name = categories[cls]
        
        annotation_data = {
            "image_id": ann.filename.split('/')[-1],
            "category_id": category_name,
            "bbox": (int(xyxy[0]), int(xyxy[1]), int(xyxy[2]), int(xyxy[3])),
            "confidence": str(conf)
        }
        predictions_data["annotations"][annos_idx] = annotation_data
        annos_idx += 1

print(f'Processed {len(anns)} test images')
print(f'Total detections: {annos_idx}')

In [ ]:
# Save predictions to writable directory
output_path = f'{WORK_DIR}/prediction.json'
with open(output_path, "w") as outfile:
    json.dump(predictions_data, outfile)
    
print(f'Predictions saved to: {output_path}')